In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 · Silver: clean, validate, deduplicate
# MAGIC
# MAGIC Reads the Bronze tables **incrementally** (streaming from Delta, `availableNow`) and writes
# MAGIC typed, trustworthy Silver tables using `foreachBatch` + `MERGE`.
# MAGIC
# MAGIC | Bronze | Silver | Pattern |
# MAGIC |---|---|---|
# MAGIC | `app_logs_raw` | `app_logs` | parse/cast → data-quality split → dedupe → insert-only MERGE on `event_id` |
# MAGIC | | `app_logs_quarantine` | rows that fail data-quality rules, with the reason and the raw record |
# MAGIC | `incident_updates_raw` | `incident_status_history` | every status change, insert-only MERGE |
# MAGIC | | `incidents` | one row per incident, latest status (SCD Type 1 upsert) |
# MAGIC | `deployments_raw` | `deployments` | typed, insert-only MERGE on `deployment_id` |
# MAGIC
# MAGIC **Why MERGE instead of append?** It makes Silver idempotent: re-running a batch, or
# MAGIC reprocessing Bronze from scratch, never creates duplicates. It also handles duplicates that
# MAGIC arrive in *different* batches, which `dropDuplicates` inside one batch can't see.

# COMMAND ----------

from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

CATALOG = "incident_analytics"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
STATE = f"/Volumes/{CATALOG}/raw/checkpoints"

# Bump RUN_ID to reprocess Silver from scratch (new checkpoints + new idempotent-write id).
# If you do, TRUNCATE the quarantine table first (the MERGE tables don't need it).
RUN_ID = "v1"

spark.conf.set("spark.sql.session.timeZone", "UTC")  # timestamps without a zone are read as UTC
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Target tables
# MAGIC Declared up front so the schema is explicit and documented, rather than inferred from the first batch.
# MAGIC `CLUSTER BY` = liquid clustering: Delta co-locates rows by these columns, so filters on date/service skip files.

# COMMAND ----------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER}.app_logs (
  event_id        STRING    NOT NULL,
  event_ts        TIMESTAMP NOT NULL COMMENT 'Parsed from three source formats, UTC',
  event_date      DATE,
  event_hour      TIMESTAMP,
  service         STRING,
  host            STRING,
  endpoint        STRING,
  trace_id        STRING,
  user_id         STRING,
  app_version     STRING,
  level           STRING    COMMENT 'Normalised to INFO / WARN / ERROR',
  status_code     INT,
  latency_ms      INT,
  error_code      STRING,
  message         STRING,
  region          STRING    COMMENT 'Added by the source from batch 4 onward; NULL before',
  raw_timestamp   STRING    COMMENT 'Original value, kept for audit',
  _rescued_data   STRING,
  _source_file    STRING,
  _ingested_at    TIMESTAMP,
  _silver_loaded_at TIMESTAMP
) CLUSTER BY (event_date, service)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER}.app_logs_quarantine (
  event_id        STRING,
  raw_timestamp   STRING,
  service         STRING,
  dq_issue        STRING COMMENT 'First rule the row failed',
  raw_record      STRING COMMENT 'All Bronze columns as JSON (NULL fields omitted)',
  _source_file    STRING,
  _ingested_at    TIMESTAMP,
  _quarantined_at TIMESTAMP
)
""")

incident_cols = """
  incident_id           STRING NOT NULL,
  service               STRING,
  severity              STRING,
  status                STRING,
  title                 STRING,
  opened_at             TIMESTAMP,
  updated_at            TIMESTAMP,
  resolved_at           TIMESTAMP,
  assigned_team         STRING,
  related_deployment_id STRING,
  _source_file          STRING,
  _ingested_at          TIMESTAMP,
  _silver_loaded_at     TIMESTAMP
"""
spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER}.incident_status_history ({incident_cols})")
spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER}.incidents ({incident_cols})")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER}.deployments (
  deployment_id     STRING NOT NULL,
  service           STRING,
  version           STRING,
  deployed_at       TIMESTAMP,
  deployed_by       STRING,
  change_type       STRING COMMENT 'release / hotfix / rollback',
  _source_file      STRING,
  _ingested_at      TIMESTAMP,
  _silver_loaded_at TIMESTAMP
)
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Logs: cleaning rules

# COMMAND ----------

LOG_COLUMNS = [f.name for f in spark.table(f"{SILVER}.app_logs").schema.fields]
VALID_LEVELS = ["INFO", "WARN", "ERROR"]


def col_or_null(df, name, dtype="string"):
    """Bronze columns appear over time (schema evolution); don't fail if one isn't there yet."""
    return F.col(name) if name in df.columns else F.lit(None).cast(dtype)


def clean_logs(df):
    raw_record = F.to_json(F.struct(*[F.col(c) for c in df.columns]))
    df = df.select(
        F.col("event_id"),
        F.trim(F.col("timestamp")).alias("raw_timestamp"),
        F.col("service"), F.col("host"), F.col("endpoint"), F.col("trace_id"),
        col_or_null(df, "user_id").alias("user_id"),
        F.col("app_version"),
        F.upper(F.trim(F.col("level"))).alias("level"),
        F.expr("try_cast(status_code AS INT)").alias("status_code"),
        F.expr("try_cast(latency_ms AS INT)").alias("latency_ms"),
        col_or_null(df, "error_code").alias("error_code"),
        F.col("message"),
        col_or_null(df, "region").alias("region"),
        col_or_null(df, "_rescued_data").alias("_rescued_data"),
        F.col("_source_file"), F.col("_ingested_at"),
        raw_record.alias("raw_record"),
    )

    # Three timestamp formats in the source:
    #   1700000000000              epoch milliseconds (13 digits)
    #   2026-10-27T09:08:22.343Z   ISO-8601 UTC
    #   2026-10-27 09:08:22        no zone -> session time zone (UTC, set above)
    # try_cast returns NULL instead of failing the job on anything else.
    event_ts = F.expr("""
        CASE WHEN raw_timestamp RLIKE '^[0-9]{13}$'
             THEN timestamp_millis(CAST(raw_timestamp AS BIGINT))
             ELSE try_cast(raw_timestamp AS TIMESTAMP)
        END""")

    level = F.when(F.col("level") == "WARNING", "WARN").otherwise(F.col("level"))

    df = (df.withColumn("event_ts", event_ts)
            .withColumn("level", level)
            .withColumn("event_date", F.to_date("event_ts"))
            .withColumn("event_hour", F.date_trunc("hour", "event_ts")))

    # First failing rule wins. Truncated JSON lines fail one of these.
    dq_issue = (F.when(F.col("event_id").isNull(), "missing_event_id")
                 .when(F.col("event_ts").isNull(), "unparseable_timestamp")
                 .when(F.col("service").isNull(), "missing_service")
                 .when(F.col("level").isNull() | ~F.col("level").isin(VALID_LEVELS), "invalid_level"))
    return df.withColumn("dq_issue", dq_issue)


def process_logs_batch(batch_df, batch_id):
    s = batch_df.sparkSession  # use the batch's session inside foreachBatch (required on serverless)
    now = F.current_timestamp()
    cleaned = clean_logs(batch_df)

    # 1) Quarantine. txnAppId + txnVersion make this append idempotent: if this batch is retried,
    #    Delta sees the same (appId, batch_id) pair and skips the duplicate write.
    (cleaned.filter(F.col("dq_issue").isNotNull())
        .select("event_id", "raw_timestamp", "service", "dq_issue", "raw_record",
                "_source_file", "_ingested_at", now.alias("_quarantined_at"))
        .write.format("delta").mode("append")
        .option("txnAppId", f"silver_logs_quarantine_{RUN_ID}")
        .option("txnVersion", batch_id)
        .saveAsTable(f"{SILVER}.app_logs_quarantine"))

    # 2) Good rows: dedupe inside the batch, then insert only event_ids Silver hasn't seen.
    good = (cleaned.filter(F.col("dq_issue").isNull())
                   .dropDuplicates(["event_id"])
                   .withColumn("_silver_loaded_at", now)
                   .select(*LOG_COLUMNS))
    (DeltaTable.forName(s, f"{SILVER}.app_logs").alias("t")
        .merge(good.alias("s"), "t.event_id = s.event_id")
        .whenNotMatchedInsertAll()
        .execute())

# COMMAND ----------

# MAGIC %md
# MAGIC ## Incidents and deployments

# COMMAND ----------

INCIDENT_COLUMNS = [f.name for f in spark.table(f"{SILVER}.incidents").schema.fields]
DEPLOYMENT_COLUMNS = [f.name for f in spark.table(f"{SILVER}.deployments").schema.fields]
STATUS_RANK = F.expr("CASE status WHEN 'OPEN' THEN 1 WHEN 'INVESTIGATING' THEN 2 WHEN 'RESOLVED' THEN 3 ELSE 0 END")


def ts(name):
    return F.expr(f"try_cast({name} AS TIMESTAMP)").alias(name)


def process_incidents_batch(batch_df, batch_id):
    s = batch_df.sparkSession
    updates = (batch_df.select(
                    F.trim("incident_id").alias("incident_id"), "service",
                    F.upper(F.trim("severity")).alias("severity"),
                    F.upper(F.trim("status")).alias("status"),
                    "title", ts("opened_at"), ts("updated_at"), ts("resolved_at"),
                    "assigned_team", "related_deployment_id", "_source_file", "_ingested_at",
                    F.current_timestamp().alias("_silver_loaded_at"))
               .filter("incident_id IS NOT NULL AND updated_at IS NOT NULL")
               .select(*INCIDENT_COLUMNS))

    # History: every distinct status change, kept forever (used for time-to-acknowledge in Gold).
    (DeltaTable.forName(s, f"{SILVER}.incident_status_history").alias("t")
        .merge(updates.dropDuplicates(["incident_id", "status", "updated_at"]).alias("s"),
               "t.incident_id = s.incident_id AND t.status = s.status AND t.updated_at = s.updated_at")
        .whenNotMatchedInsertAll()
        .execute())

    # Current state: one batch can hold OPEN *and* INVESTIGATING for the same incident,
    # so keep only the latest update per incident before merging.
    w = Window.partitionBy("incident_id").orderBy(F.col("updated_at").desc(), STATUS_RANK.desc())
    latest = updates.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

    # Only overwrite with a *newer* update, so a late file can't move RESOLVED back to OPEN.
    (DeltaTable.forName(s, f"{SILVER}.incidents").alias("t")
        .merge(latest.alias("s"), "t.incident_id = s.incident_id")
        .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
        .whenNotMatchedInsertAll()
        .execute())


def process_deployments_batch(batch_df, batch_id):
    s = batch_df.sparkSession
    deps = (batch_df.select(
                F.trim("deployment_id").alias("deployment_id"), "service", "version",
                ts("deployed_at"), "deployed_by",
                F.lower(F.trim("change_type")).alias("change_type"),
                "_source_file", "_ingested_at",
                F.current_timestamp().alias("_silver_loaded_at"))
            .filter("deployment_id IS NOT NULL AND deployed_at IS NOT NULL")
            .dropDuplicates(["deployment_id"])
            .select(*DEPLOYMENT_COLUMNS))
    (DeltaTable.forName(s, f"{SILVER}.deployments").alias("t")
        .merge(deps.alias("s"), "t.deployment_id = s.deployment_id")
        .whenNotMatchedInsertAll()
        .execute())

# COMMAND ----------

# MAGIC %md
# MAGIC ## Run: stream new Bronze rows through each function, then stop

# COMMAND ----------

def run_silver(bronze_table, batch_fn, name):
    query = (spark.readStream.table(f"{BRONZE}.{bronze_table}")
             .writeStream
             .foreachBatch(batch_fn)
             .option("checkpointLocation", f"{STATE}/silver_{name}_{RUN_ID}/_checkpoint")
             .trigger(availableNow=True)
             .start())
    query.awaitTermination()
    print(f"bronze.{bronze_table} -> silver ({name}): done")


run_silver("app_logs_raw", process_logs_batch, "logs")
run_silver("incident_updates_raw", process_incidents_batch, "incidents")
run_silver("deployments_raw", process_deployments_batch, "deployments")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Checks

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Reconciliation: every Bronze row is accounted for
# MAGIC SELECT
# MAGIC   b.bronze_rows,
# MAGIC   s.silver_rows,
# MAGIC   q.quarantined_rows,
# MAGIC   b.bronze_rows - s.silver_rows - q.quarantined_rows AS duplicates_removed
# MAGIC FROM (SELECT COUNT(*) AS bronze_rows FROM incident_analytics.bronze.app_logs_raw) b,
# MAGIC      (SELECT COUNT(*) AS silver_rows FROM incident_analytics.silver.app_logs) s,
# MAGIC      (SELECT COUNT(*) AS quarantined_rows FROM incident_analytics.silver.app_logs_quarantine) q;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Should be exactly three levels, no NULL timestamps, no duplicate event_ids
# MAGIC SELECT level, COUNT(*) AS n,
# MAGIC        SUM(CASE WHEN event_ts IS NULL THEN 1 ELSE 0 END) AS null_ts
# MAGIC FROM incident_analytics.silver.app_logs GROUP BY level;

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_event_ids FROM incident_analytics.silver.app_logs;

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT dq_issue, COUNT(*) AS n FROM incident_analytics.silver.app_logs_quarantine GROUP BY dq_issue;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Each format parsed to the same kind of value
# MAGIC SELECT
# MAGIC   CASE WHEN raw_timestamp RLIKE '^[0-9]{13}$' THEN 'epoch_millis'
# MAGIC        WHEN raw_timestamp LIKE '%T%Z'         THEN 'iso_utc'
# MAGIC        ELSE 'no_zone' END AS source_format,
# MAGIC   COUNT(*) AS n, MIN(event_ts) AS min_ts, MAX(event_ts) AS max_ts
# MAGIC FROM incident_analytics.silver.app_logs GROUP BY 1;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Current incident state vs the full status trail
# MAGIC SELECT i.incident_id, i.service, i.severity, i.status AS current_status,
# MAGIC        i.opened_at, i.resolved_at, i.related_deployment_id,
# MAGIC        TRANSFORM(ARRAY_SORT(COLLECT_LIST(STRUCT(h.updated_at, h.status))), x -> x.status) AS status_trail
# MAGIC FROM incident_analytics.silver.incidents i
# MAGIC JOIN incident_analytics.silver.incident_status_history h USING (incident_id)
# MAGIC GROUP BY ALL ORDER BY i.opened_at;